# MediDiary — Bengali Whisper setup & benchmark

MediDiary's speech model is **[SayedShaun/bengali-whisper-medium-ggml](https://huggingface.co/SayedShaun/bengali-whisper-medium-ggml)**
(Apache-2.0; whisper.cpp conversion of `SayedShaun/bengali-whisper-medium`, fine-tuned from `openai/whisper-medium`).
It is already in whisper.cpp's ggml format, so **no conversion is needed** — this notebook just helps you pick the right
quantisation and produces the file the app expects.

| file | size | use |
|---|---|---|
| `ggml-model-q6_k.bin` | 610 MB | **default** — best quality per byte |
| `ggml-model-q4_0.bin` | 424 MB | faster / smaller — use if q6_k is too slow on the demo phone |
| `ggml-model-q2_k.bin` | 254 MB | not recommended — the author warns it can hallucinate or drop words |

**What this notebook does**
1. Downloads the chosen quantisations and checks the file header (same check the app does on import).
2. Builds `whisper-cli` from **whisper.cpp v1.9.5** — the exact version compiled into the app — so if it runs here, it loads on the phone.
3. Transcribes your Bangla test clips with every quantisation and reports **CER** (if you give reference text) and **real-time factor**.
4. Saves the winner to Drive as **`medidiary-whisper.bin`**.

Runtime: any Colab runtime (CPU). ≈ 10 min + ~1 min per clip per model.

In [ ]:
REPO        = "SayedShaun/bengali-whisper-medium-ggml"
QUANTS      = ["q6_k", "q4_0"]
THREADS     = 4
LANGUAGE    = "bn"
WHISPER_CPP_TAG = "v1.9.5"
DRIVE_OUT   = "/content/drive/MyDrive/medidiary_whisper"

REFERENCES = {
}

import os
if DRIVE_OUT:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
%%capture
!pip install -q huggingface_hub
!git clone -q --depth 1 --branch {WHISPER_CPP_TAG} https://github.com/ggml-org/whisper.cpp /content/whisper.cpp

In [ ]:
from huggingface_hub import hf_hub_download

MODELS = {}
for q in QUANTS:
    path = hf_hub_download(REPO, f"ggml-model-{q}.bin", local_dir = "/content/models")
    with open(path, "rb") as f:
        magic = f.read(4)
    assert magic == b"lmgg", f"{q}: unexpected header {magic!r}"
    MODELS[q] = path
    print(f"{q:6s} {os.path.getsize(path)/1e6:6.0f} MB  ok  {path}")

In [ ]:
!cmake -S /content/whisper.cpp -B /content/whisper.cpp/build -DCMAKE_BUILD_TYPE=Release -DWHISPER_BUILD_TESTS=OFF > /dev/null
!cmake --build /content/whisper.cpp/build -j --target whisper-cli 2>&1 | tail -1
CLI = "/content/whisper.cpp/build/bin/whisper-cli"
assert os.path.exists(CLI)

In [ ]:
import glob, subprocess, wave
from google.colab import files

uploaded = list(files.upload().keys())
CLIPS = {}
os.makedirs("/content/clips16k", exist_ok = True)
for name in uploaded:
    out = f"/content/clips16k/{os.path.splitext(name)[0]}.wav"
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", name, "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", out], check = True)
    with wave.open(out) as w:
        dur = w.getnframes() / w.getframerate()
    CLIPS[name] = (out, dur)
    print(f"{name}: {dur:.1f} s" + ("  (has reference)" if name in REFERENCES else ""))
if any(d > 60 for _, d in CLIPS.values()):
    print("note: the app records at most 60 s per check-in")

In [ ]:
import re, time, unicodedata

def normalize(t):
    t = unicodedata.normalize("NFC", t).lower()
    t = re.sub(r"\[.*?\]|\(.*?\)", " ", t)
    t = re.sub(r"[।,.!?;:\"'“”‘’\-–—…]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(ref, hyp):
    r, h = normalize(ref), normalize(hyp)
    return edit_distance(r, h) / max(1, len(r))

def transcribe(model, wav):
    t0 = time.perf_counter()
    r = subprocess.run([CLI, "-m", model, "-f", wav, "-l", LANGUAGE, "-nt", "-t", str(THREADS)],
                       capture_output = True, text = True)
    wall = time.perf_counter() - t0
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1500:])
    text = " ".join(l.strip() for l in r.stdout.splitlines() if l.strip())
    m = re.search(r"total time\s*=\s*([\d.]+)\s*ms", r.stderr)
    return text, (float(m.group(1)) / 1000 if m else wall)

RESULTS = {q: [] for q in MODELS}
for name, (wav, dur) in CLIPS.items():
    print(f"\n▶ {name} ({dur:.1f} s)")
    if name in REFERENCES: print(f"  REF   : {REFERENCES[name]}")
    for q, path in MODELS.items():
        text, secs = transcribe(path, wav)
        c = cer(REFERENCES[name], text) if name in REFERENCES else None
        RESULTS[q].append({"clip": name, "dur": dur, "secs": secs, "cer": c, "text": text})
        print(f"  {q:6s}: {text}\n          {secs:.1f} s  (RTF {secs/dur:.2f})" + (f"  CER {c:.1%}" if c is not None else ""))

print("\n=== Summary ===")
for q, rows in RESULTS.items():
    rtf = sum(r["secs"] for r in rows) / max(1e-9, sum(r["dur"] for r in rows))
    cers = [r["cer"] for r in rows if r["cer"] is not None]
    print(f"{q:6s}  RTF {rtf:.2f}" + (f"   mean CER {sum(cers)/len(cers):.1%} on {len(cers)} clips" if cers else "   (no references → no CER)"))
print("\nRTF < 1 means faster than real time. Colab's x86 CPU is NOT a phone: use this to compare\n"
      "quantisations, then confirm speed on the demo phone (transcript screen shows on-device time).")

In [ ]:
import shutil
CHOSEN = QUANTS[0]
APP_NAME = "medidiary-whisper.bin"
shutil.copy(MODELS[CHOSEN], f"/content/{APP_NAME}")
print(f"/content/{APP_NAME}  ← {CHOSEN}")
if DRIVE_OUT:
    os.makedirs(DRIVE_OUT, exist_ok = True)
    shutil.copy(MODELS[CHOSEN], f"{DRIVE_OUT}/{APP_NAME}")
    print("saved to", f"{DRIVE_OUT}/{APP_NAME}")

## Put it on the phone

**Option A — adb** (after installing and opening the app once):
```bash
adb push medidiary-whisper.bin /sdcard/Android/data/com.medidiary.app/files/models/medidiary-whisper.bin
```
**Option B — in the app:** download `medidiary-whisper.bin` from Drive on the phone → **Settings → Load model → Import…**

Then **Settings → Load model → Re-scan**, and keep the main-screen language toggle on **BN**.

## Check it on the demo phone
* Record a 20–30 s Bangla check-in in airplane mode. The transcript screen shows **"Transcribed on-device in X s"**.
* If that is too slow for the live demo, repeat with `CHOSEN = "q4_0"`.
* Whisper and Gemma are never loaded at the same time, so peak RAM ≈ the larger of the two models plus working buffers.

## Attribution (already in the app's README and About screen)
Speech model: SayedShaun/bengali-whisper-medium-ggml (Apache-2.0), converted from SayedShaun/bengali-whisper-medium,
fine-tuned from openai/whisper-medium (MIT). Runtime: whisper.cpp (MIT).